# EDA — Vietnam-Celeb-T for ECAPA fine-tuning
This notebook downloads the official Vietnam-Celeb release and analyses the exact `train.csv`, `val.csv`, and speaker-disjoint `dev.csv` generated by this project's subset-preparation script. It deliberately does **not** use Vietnam-Celeb-E/H test trials.

In [ ]:
# Run in Google Colab. Clone the same project code used for fine-tuning.
!git clone -b feature/finetune-colab https://github.com/hieuvous/Secure-Virtual-Assistant-with-Speaker-Recognition.git
%cd /content/Secure-Virtual-Assistant-with-Speaker-Recognition
!pip install -q gdown pandas matplotlib seaborn soundfile

## Download the official release
The official release is split into four zip parts. This may require substantial temporary Colab disk space; download only when the runtime has enough free space.

In [ ]:
# Official Vietnam-Celeb Interspeech release links. Files stay in /content, not Google Drive.
!gdown --fuzzy 'https://drive.google.com/file/d/1pMuT3DFzSwib7SVcRS8VkDwPuLTsemSG/view?usp=share_link'
!gdown --fuzzy 'https://drive.google.com/file/d/1xayHt2HRqE1aJ4HvtUT40_9XlgvfDfRY/view?usp=share_link'
!gdown --fuzzy 'https://drive.google.com/file/d/1MIlM78EbN_J9cApkNes_2_BrFrf8XwMc/view?usp=share_link'
!gdown --fuzzy 'https://drive.google.com/file/d/1h6Na58DC03p-502B9QpC5Z_FadUwAdNA/view?usp=share_link'
!ls -lh

In [ ]:
# Assemble and extract exactly as documented by the official dataset repository.
# If downloaded filenames differ, rename the first zip part to vietnam-celeb-part.zip first.
!zip -F vietnam-celeb-part.zip --out full-dataset.zip
!unzip -q full-dataset.zip -d /content/Vietnam-Celeb
!find /content/Vietnam-Celeb -maxdepth 2 -type f | head -20

In [ ]:
# Locate the extracted official training list and its audio folder.
from pathlib import Path

DATASET_DIR = Path('/content/Vietnam-Celeb')
OFFICIAL_TRAIN_LIST = next(DATASET_DIR.rglob('vietnam-celeb-t.txt'))
DATA_ROOT = OFFICIAL_TRAIN_LIST.parent / 'data'
assert DATA_ROOT.is_dir(), f'Expected speaker folders at {DATA_ROOT}'
print('Official train list:', OFFICIAL_TRAIN_LIST)
print('Audio root:', DATA_ROOT)

In [ ]:
# Inspect the official Vietnam-Celeb-T list used to restrict the training pool.
from training.prepare_vietnam_celeb_subset import speakers_from_train_list

official_rows = OFFICIAL_TRAIN_LIST.read_text(encoding='utf-8', errors='ignore').splitlines()
official_speakers = speakers_from_train_list(OFFICIAL_TRAIN_LIST)
print('Vietnam-Celeb-T list rows:', len(official_rows))
print('Vietnam-Celeb-T speakers parsed:', len(official_speakers))
print('First five rows:')
print(*official_rows[:5], sep='\n')

In [ ]:
# Build the same metadata consumed by finetune_ecapa_colab.py.
# Change these values only if your actual experiment uses different values.
METADATA_DIR = Path('/content/SpeakerRecognition/metadata_eda')
TRAIN_SPEAKERS = 300
DEV_SPEAKERS = 50
MAX_UTTS = 20
SEED = 42
!python training/prepare_vietnam_celeb_subset.py --data-root "$DATA_ROOT" --official-train-list "$OFFICIAL_TRAIN_LIST" --output-dir "$METADATA_DIR" --train-speakers $TRAIN_SPEAKERS --dev-speakers $DEV_SPEAKERS --max-utts $MAX_UTTS --seed $SEED

In [ ]:
# Load the exact model-input CSVs and check split integrity.
import json
import pandas as pd

splits = {name: pd.read_csv(METADATA_DIR / f'{name}.csv') for name in ('train', 'val', 'dev')}
summary = json.loads((METADATA_DIR / 'split_summary.json').read_text(encoding='utf-8'))
overview = pd.DataFrame([
    {'split': name, 'audio_rows': len(df), 'speakers': df.speaker_id.nunique(),
     'unique_paths': df.path.nunique()}
    for name, df in splits.items()
]).set_index('split')
display(overview)
train_val_speakers = set(splits['train'].speaker_id) | set(splits['val'].speaker_id)
dev_speakers = set(splits['dev'].speaker_id)
assert set(splits['train'].speaker_id) == set(splits['val'].speaker_id)
assert train_val_speakers.isdisjoint(dev_speakers)
assert summary['speaker_disjoint_train_vs_dev']
print('Split integrity: OK')
display(splits['train'].head())

In [ ]:
# Utterances per speaker: this reveals the effective impact of MAX_UTTS.
import matplotlib.pyplot as plt
import seaborn as sns

counts = pd.concat([
    df.assign(split=name).groupby(['split', 'speaker_id']).size().rename('utterances')
    for name, df in splits.items()
]).reset_index()
display(counts.groupby('split')['utterances'].describe())
plt.figure(figsize=(9, 4))
sns.histplot(data=counts, x='utterances', hue='split', multiple='dodge', discrete=True)
plt.title('Utterances per speaker in the actual fine-tuning splits')
plt.tight_layout()
plt.show()

In [ ]:
# Audio duration EDA uses headers only; unreadable files are reported rather than silently dropped.
import soundfile as sf

all_rows = pd.concat([df.assign(split=name) for name, df in splits.items()], ignore_index=True)
def audio_info(path):
    try:
        info = sf.info(path)
        return pd.Series({'duration_seconds': info.duration, 'sample_rate': info.samplerate, 'read_error': None})
    except Exception as exc:
        return pd.Series({'duration_seconds': None, 'sample_rate': None, 'read_error': str(exc)})

audio_stats = all_rows.join(all_rows.path.apply(audio_info))
print('Unreadable audio files:', int(audio_stats.read_error.notna().sum()))
display(audio_stats.groupby('split')['duration_seconds'].describe())
display(audio_stats.sample(min(10, len(audio_stats)), random_state=SEED))
plt.figure(figsize=(9, 4))
sns.histplot(data=audio_stats.dropna(subset=['duration_seconds']), x='duration_seconds', hue='split', bins=40, element='step')
plt.xlim(0, audio_stats.duration_seconds.quantile(0.99))
plt.title('Audio duration distribution (99th percentile view)')
plt.tight_layout()
plt.show()

In [ ]:
# Listen to one file from the exact train split (optional).
from IPython.display import Audio, display
sample_path = splits['train'].sample(1, random_state=SEED).iloc[0].path
print(sample_path)
display(Audio(sample_path))

## Interpretation checklist
- Report counts from `overview`, not counts from Vietnam-Celeb-E/H.
- Keep `SEED`, `TRAIN_SPEAKERS`, `DEV_SPEAKERS`, and `MAX_UTTS` identical to the fine-tune command when making report figures.
- The resulting `METADATA_DIR/train.csv` and `val.csv` are the files to pass to `finetune_ecapa_colab.py`; `dev.csv` is for verification evaluation.